# Descargar historia BCCh desde 2004

Este notebook descarga la historia completa de tasas BCCh desde `2004-01-01` usando el catálogo actual del proyecto.

Series incluidas por defecto:

- `TPM`
- `SPC_03Y`
- `SPC_06Y`
- `SPC_1Y`
- `SPC_2Y`
- `SPC_3Y`
- `SPC_4Y`
- `SPC_5Y`
- `SPC_10Y`


In [ ]:
from __future__ import annotations

import os
import sys
from datetime import date
from getpass import getpass
from pathlib import Path

import pandas as pd

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if load_dotenv is not None:
    load_dotenv(PROJECT_ROOT / ".env")

from yield_curve import DEFAULT_NS_COLUMNS, RATE_SERIES, fetch_bcch_series, prepare_rates_dataframe

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


In [ ]:
START_DATE = "2004-01-01"
END_DATE = date.today().isoformat()
SERIES_KEYS = DEFAULT_NS_COLUMNS.copy()

print("Proyecto:", PROJECT_ROOT)
print("Rango:", START_DATE, "->", END_DATE)
print("Series:", ", ".join(SERIES_KEYS))
display(pd.DataFrame([
    {
        "key": key,
        "label": RATE_SERIES[key].label,
        "code": RATE_SERIES[key].code,
        "months": RATE_SERIES[key].months,
    }
    for key in SERIES_KEYS
]))


In [ ]:
BCCH_USER = os.getenv("BCCH_USER") or input("Usuario BCCh: ").strip()
BCCH_PASSWORD = os.getenv("BCCH_PASSWORD") or getpass("Contrasena BCCh: ")

if not BCCH_USER or not BCCH_PASSWORD:
    raise ValueError("Debes ingresar credenciales BCCh.")


In [ ]:
raw_df = fetch_bcch_series(
    series_keys=SERIES_KEYS,
    user=BCCH_USER,
    password=BCCH_PASSWORD,
    start_date=START_DATE,
    end_date=END_DATE,
)

clean_df = prepare_rates_dataframe(raw_df)

summary = pd.DataFrame([
    {
        "start_date_requested": START_DATE,
        "end_date_requested": END_DATE,
        "raw_rows": len(raw_df),
        "clean_rows": len(clean_df),
        "removed_rows": len(raw_df) - len(clean_df),
        "first_raw_date": raw_df["Date"].min(),
        "last_raw_date": raw_df["Date"].max(),
        "first_clean_date": clean_df["Date"].min(),
        "last_clean_date": clean_df["Date"].max(),
    }
])

summary


In [ ]:
display(raw_df.head(10))
display(clean_df.head(10))


In [ ]:
output_dir = PROJECT_ROOT / "outputs"
output_dir.mkdir(exist_ok=True)

raw_path = output_dir / "bcch_rates_raw_from_2004.csv"
clean_path = output_dir / "bcch_rates_clean_from_2004.csv"

raw_export = raw_df.copy()
clean_export = clean_df.copy()
raw_export["Date"] = pd.to_datetime(raw_export["Date"]).dt.strftime("%Y-%m-%d")
clean_export["Date"] = pd.to_datetime(clean_export["Date"]).dt.strftime("%Y-%m-%d")

raw_export.to_csv(raw_path, index=False)
clean_export.to_csv(clean_path, index=False)

print("Archivo raw:", raw_path)
print("Archivo limpio:", clean_path)
